# PCA bases - statistics
03.27.2026

How do each of these basis compare? RMS, self response, etc. 

Interested in comparing how the amplitude and sparkle angle impact these values.

Also contributes to knowing which claibrations have a flat as the dominant mode. 

In [14]:
# function to go through each basis, calculate the RMS, self RMS, and ratio of self RMS to RMS
calib_dir = "/home/eden/data/spark_calib"
import sparkles.calibration_iter as calibi 
from importlib import reload

In [15]:
reload(calibi)

df = calibi.calibration_rms_df("/home/eden/data/spark_calib")

/tmp/ipykernel_1580841/1609268561.py:3: UserWarning: Skipping sep22_ang60_amp0.030_freq2000: missing ref_rms.fits
  df = calibi.calibration_rms_df("/home/eden/data/spark_calib")


In [16]:
df

,calib_folder,calib_path,spark_sep,spark_ang,spark_amp,wfs_hz,pca_mode,rms,pca_rms
0,sep10_ang00_amp0.020_freq2000,/home/eden/data/spark_calib/sep10_ang00_amp0.0...,10.0,0.0,0.02,2000.0,0,0.001456,0.008333
1,sep10_ang00_amp0.020_freq2000,/home/eden/data/spark_calib/sep10_ang00_amp0.0...,10.0,0.0,0.02,2000.0,1,0.001350,0.008333
2,sep10_ang00_amp0.020_freq2000,/home/eden/data/spark_calib/sep10_ang00_amp0.0...,10.0,0.0,0.02,2000.0,2,0.000030,0.008333
3,sep10_ang00_amp0.030_freq2000,/home/eden/data/spark_calib/sep10_ang00_amp0.0...,10.0,0.0,0.03,2000.0,0,0.002141,0.008333
4,sep10_ang00_amp0.030_freq2000,/home/eden/data/spark_calib/sep10_ang00_amp0.0...,10.0,0.0,0.03,2000.0,1,0.001975,0.008333
...,...,...,...,...,...,...,...,...,...
250,sep22_ang90_amp0.030_freq2000,/home/eden/data/spark_calib/sep22_ang90_amp0.0...,22.0,90.0,0.03,2000.0,1,0.000674,0.008333
251,sep22_ang90_amp0.030_freq2000,/home/eden/data/spark_calib/sep22_ang90_amp0.0...,22.0,90.0,0.03,2000.0,2,0.000627,0.008333
252,sep22_ang90_amp0.040_freq2000,/home/eden/data/spark_calib/sep22_ang90_amp0.0...,22.0,90.0,0.04,2000.0,0,0.000026,0.008333
253,sep22_ang90_amp0.040_freq2000,/home/eden/data/spark_calib/sep22_ang90_amp0.0...,22.0,90.0,0.04,2000.0,1,0.000844,0.008333


In [ ]:
# For a given sparkle separation, plot angle/amplitude impact on RMS.
# One heatmap per PCA mode.

import numpy as np
import matplotlib.pyplot as plt


def plot_rms_by_mode_grid(df, spark_sep, value_col="rms", cmap="viridis"):
    sub = df[df["spark_sep"] == spark_sep].copy()
    if sub.empty:
        raise ValueError(f"No rows found for spark_sep={spark_sep}")

    modes = sorted(sub["pca_mode"].unique())
    n_modes = len(modes)
    n_cols = min(3, n_modes)
    n_rows = int(np.ceil(n_modes / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.8 * n_cols, 4.0 * n_rows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()

    vmin = sub[value_col].min()
    vmax = sub[value_col].max()

    for i, mode in enumerate(modes):
        ax = axes[i]
        mode_df = sub[sub["pca_mode"] == mode]
        grid = mode_df.pivot_table(
            index="spark_ang", columns="spark_amp", values=value_col, aggfunc="mean"
        ).sort_index(axis=0).sort_index(axis=1)

        im = ax.imshow(
            grid.values,
            aspect="auto",
            origin="lower",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(f"PCA mode {mode}")
        ax.set_xlabel("spark_amp")
        ax.set_ylabel("spark_ang")
        ax.set_xticks(np.arange(len(grid.columns)))
        ax.set_xticklabels([f"{x:.3f}" for x in grid.columns], rotation=45, ha="right")
        ax.set_yticks(np.arange(len(grid.index)))
        ax.set_yticklabels([f"{y:.0f}" for y in grid.index])

    for j in range(n_modes, len(axes)):
        axes[j].axis("off")

    cbar = fig.colorbar(im, ax=axes[:n_modes], shrink=0.95)
    cbar.set_label(value_col)
    fig.suptitle(f"Separation {spark_sep}: {value_col} by angle/amplitude", y=1.02)
    return fig, axes


# Example usage:
plot_rms_by_mode_grid(df, spark_sep=22, value_col="rms")
# plot_rms_by_mode_grid(df, spark_sep=22, value_col="pca_rms")
plt.show()

